# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a detailed template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Title:", metadata.get('name', 'Name not found'))
print("Description:", metadata.get('description', 'Description not found'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This section will enumerate the record sets (tables) in the dataset, their fields and columns, referencing all by their `@id` for consistent handling.

In [ ]:
# List record sets and their fields' @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for record_set in record_sets:
        rs_id = getattr(record_set, '@id', None)
        rs_name = getattr(record_set, 'name', None)
        print(f"RecordSet @id: {rs_id} | Name: {rs_name}")
        fields = getattr(record_set, 'fields', [])
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            print(f"    Field @id: {field_id} | Field Name: {field_name}")
        print("")
        columns = getattr(record_set, 'columns', [])
        for column in columns:
            col_id = getattr(column, '@id', None)
            col_name = getattr(column, 'name', None)
            print(f"    Column @id: {col_id} | Column Name: {col_name}")
        print("-----------------------------------")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field/column `@id`s from the overview.

For this section, we'll extract all available record sets, referencing by their `@id`, and show the resulting DataFrame structures.

In [ ]:
# Extract data from each record set (referenced by @id)
dataframes = {}
record_set_ids = [getattr(rs, '@id') for rs in dataset.metadata.record_sets] if dataset.metadata.record_sets else []
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for RecordSet {rs_id} loaded with shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use the `@id` references found above for field selection. For illustration, choose the first loaded record set and the first numeric field.

In [ ]:
from numpy import number

# Pick the first record set and try to select a numeric field
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]

    # Find first numeric field or column by @id
    numeric_col_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col_id = col
            break
    if numeric_col_id is not None:
        print(f"Using numeric field for EDA: {numeric_col_id}")

        threshold = 10
        filtered_df = df[df[numeric_col_id] > threshold]
        print(f"Filtered records with {numeric_col_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_col_id}_normalized"] = (
            filtered_df[numeric_col_id] - filtered_df[numeric_col_id].mean()
        ) / filtered_df[numeric_col_id].std()

        print(f"Normalized {numeric_col_id} for filtered records:")
        print(filtered_df[[numeric_col_id, f"{numeric_col_id}_normalized"]].head())

        # Try to group by a categorical column
        group_col_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_col_id:
                group_col_id = col
                break
        if group_col_id:
            print(f"Grouping records by {group_col_id}:")
            grouped_df = filtered_df.groupby(group_col_id)[numeric_col_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical column found for grouping.")
    else:
        print("No numeric column found for EDA.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of the numeric field and grouped averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if available
if dataframes and numeric_col_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_col_id], kde=True)
    plt.title(f"Distribution of {numeric_col_id}")
    plt.xlabel(numeric_col_id)
    plt.ylabel("Count")
    plt.show()

    if group_col_id and 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_col_id, y=numeric_col_id, data=grouped_df)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_col_id} by {group_col_id}")
        plt.xlabel(group_col_id)
        plt.ylabel(f"Mean {numeric_col_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated the workflow for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`. Key steps included:
- Accessing metadata and dataset structure via Croissant schema.
- Enumerating record sets, fields, and columns using `@id`.
- Extracting data into DataFrames and conducting basic EDA (filtering, normalization, grouping).
- Visualizing numeric distributions and relationships.

This approach ensures reproducibility and schema-based referencing for downstream analysis.